In [26]:
%load_ext autoreload
%autoreload 2

import open3d as o3d
import numpy as np
import os
import sys
from glob import glob
import pickle
import json
from pathlib import Path
from copy import deepcopy

import sys
sys.path.append('D:\Dev\Point-Cloud-Registration\source')
from utils.write_output import write_array2txt

from cylinder_extraction.cylinder import Cylinder
from cylinder_extraction.station import Station
from registration.utils import multiway_load_point_clouds as load_point_clouds
from registration.utils import multiway_full_registration_with_sequence as full_registration_with_sequence
from registration.utils import multiway_full_registration_with_pairs as full_registration_with_pairs
from registration.utils import pairwise_registration, combine_point_cloud, multiway_filter_point_clouds

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Input
The first part of the tutorial code reads three point clouds from files. The point clouds are downsampled and visualized together. They are misaligned.

In [27]:
root_dir = Path('D:/REGISTRATION DATA/External/JB-595 117ft EXT - FARO')
data_dir = root_dir / 'target_tank' / 'global' / 'ground'

In [28]:
with open(os.path.join(root_dir, 'globalization_axis_phase_1.pkl'), 'rb') as f:
    bundle = pickle.load(f)
all_cylinders = bundle.target_tank

In [29]:
# with open(root_dir / 'tank_info.pkl', 'rb') as f:
#     tank_info = pickle.load(f)
# all_cylinders = tank_info['tanks'][tank_label]


In [30]:
in_cyls, out_cyls = multiway_filter_point_clouds(all_cylinders, radius_offset_max=0.5)

In [ ]:
out_cyls

[]

### Read point clouds for inlier cylinders

In [31]:
# with open(os.path.join(root_dir, 'thin_slice', 'station_order.txt'), 'r') as f:
#     order = json.load(f)
# order = order['order']
# print(order)

# read pairs
with open(root_dir / 'target_tank' / 'global' / 'thin_slice' / 'station_pairs.txt', 'r') as f:
    pair_info = json.load(f)
pair_info = pair_info['pair']
pair_info = [[p0-4, p1-4] for p0,p1 in pair_info]
print(pair_info)
# order = range(len(in_cyls))

[[1, 2], [13, 12], [14, 15], [8, 9], [9, 10], [2, 3], [13, 14], [6, 7], [11, 12], [7, 8], [3, 4], [10, 11], [4, 5], [5, 6], [0, 1]]


In [32]:
# all_pcds_down, all_pcds_full = load_point_clouds([in_cyls[i] for i in [0,1,2,3,4,5,6,9,10,7,8,11,13,14,12]], file_dir=data_dir)
all_pcds_down, all_pcds_full = load_point_clouds(in_cyls, file_dir=data_dir, voxel_size=0.01)
# pcds_down, pcds_full = load_point_clouds([in_cyls[i] for i in [0,1]], file_dir=data_dir)
# in_cyls.pop(3)
# pcds_down, pcds_full = load_point_clouds(in_cyls, file_dir=data_dir, radius=0.05, max_nn=10)

D:\REGISTRATION DATA\External\JB-595 117ft EXT - FARO\target_tank\global\ground\807_Scan_001 cluster_5.pcd
D:\REGISTRATION DATA\External\JB-595 117ft EXT - FARO\target_tank\global\ground\807_Scan_002 cluster_6.pcd
D:\REGISTRATION DATA\External\JB-595 117ft EXT - FARO\target_tank\global\ground\807_Scan_003 cluster_9.pcd
D:\REGISTRATION DATA\External\JB-595 117ft EXT - FARO\target_tank\global\ground\807_Scan_004 cluster_3.pcd
D:\REGISTRATION DATA\External\JB-595 117ft EXT - FARO\target_tank\global\ground\807_Scan_005 cluster_5.pcd
D:\REGISTRATION DATA\External\JB-595 117ft EXT - FARO\target_tank\global\ground\807_Scan_006 cluster_4.pcd
D:\REGISTRATION DATA\External\JB-595 117ft EXT - FARO\target_tank\global\ground\807_Scan_007 cluster_5.pcd
D:\REGISTRATION DATA\External\JB-595 117ft EXT - FARO\target_tank\global\ground\807_Scan_008 cluster_5.pcd
D:\REGISTRATION DATA\External\JB-595 117ft EXT - FARO\target_tank\global\ground\807_Scan_009 cluster_8.pcd
D:\REGISTRATION DATA\External\JB-595 

In [33]:
pcds_down = all_pcds_down[4:]
pcds_full = all_pcds_full[4:]

# TEST 2-LEVEL ICP

In [34]:
max_correspondence_distance_coarse = 0.2#voxel_size * 15
max_correspondence_distance_fine1  = 0.01#voxel_size * 1.5
max_correspondence_distance_fine2  = 0.005#voxel_size * 1.5
pcd_pairs = []
for i in range(0, len(pcds_down), 2):
    # icp i & i+1
    with o3d.utility.VerbosityContextManager(
            o3d.utility.VerbosityLevel.Debug) as cm:
        pose_graph = full_registration_with_pairs(
            pcds_down[i:i+2], [[0,1]],
            max_correspondence_distance_coarse,
            max_correspondence_distance_fine1,
            max_correspondence_distance_fine2
        )
    pcds_down[i+1].transform(pose_graph.nodes[1].pose)
    # combine pcd_i & pcd_i+1
    _pcd_combined = o3d.geometry.PointCloud()
    _pcd_combined.points = o3d.utility.Vector3dVector(
        np.r_[
            np.asarray(pcds_down[i].points),
            np.asarray(pcds_down[i+1].points)
        ]
    )
    _pcd_combined.normals = o3d.utility.Vector3dVector(
        np.r_[
            np.asarray(pcds_down[i].normals),
            np.asarray(pcds_down[i+1].normals)
        ]
    )
    pcd_pairs.append(_pcd_combined)



----SOURCE, TARGET = 0, 1
Coarse alignment
Apply point-to-plane ICP
[Open3D DEBUG] ICP Iteration #0: Fitness 0.1195, RMSE 0.1201
[Open3D DEBUG] Residual : 9.05e-03 (# of elements : 303697)
[Open3D DEBUG] ICP Iteration #1: Fitness 0.6317, RMSE 0.1381
[Open3D DEBUG] Residual : 1.79e-02 (# of elements : 1605818)
[Open3D DEBUG] ICP Iteration #2: Fitness 0.8346, RMSE 0.0392
[Open3D DEBUG] Residual : 5.11e-04 (# of elements : 2121613)
[Open3D DEBUG] ICP Iteration #3: Fitness 0.8307, RMSE 0.0350
[Open3D DEBUG] Residual : 1.87e-04 (# of elements : 2111784)
[Open3D DEBUG] ICP Iteration #4: Fitness 0.8292, RMSE 0.0347
[Open3D DEBUG] Residual : 1.59e-04 (# of elements : 2107810)
[Open3D DEBUG] ICP Iteration #5: Fitness 0.8284, RMSE 0.0346
[Open3D DEBUG] Residual : 1.53e-04 (# of elements : 2105901)
[Open3D DEBUG] ICP Iteration #6: Fitness 0.8281, RMSE 0.0347
[Open3D DEBUG] Residual : 1.52e-04 (# of elements : 2104997)
[Open3D DEBUG] ICP Iteration #7: Fitness 0.8279, RMSE 0.0347
[Open3D DEBUG] Res

In [ ]:
# write_dir = data_dir / 'icp_pair'
# if not os.path.exists(write_dir):
#     os.mkdir(write_dir)
# for i,pcd in enumerate(pcd_pairs):
#     o3d.io.write_point_cloud(
#         str(write_dir / f'{2*i}_{2*i+1}.pcd'),
#         pcd
#     )

In [39]:
pcd_pairs_copy = deepcopy(pcd_pairs)

In [48]:
pcd_pairs = pcd_pairs_copy
len(pcd_pairs)

8

In [49]:
pcd_pairs.append(pcd_pairs[0])

In [50]:
print("Full registration ...")
max_correspondence_distance_coarse = 0.2#voxel_size * 15
max_correspondence_distance_fine1  = 0.01#voxel_size * 1.5
max_correspondence_distance_fine2  = 0.005#voxel_size * 1.5
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    pose_graph = full_registration_with_pairs(
        pcd_pairs, [[i,i+1] for i in range(1, len(pcd_pairs))],
        max_correspondence_distance_coarse,
        max_correspondence_distance_fine1,
        max_correspondence_distance_fine2
    )

Full registration ...
----SOURCE, TARGET = 1, 2
Coarse alignment
Apply point-to-plane ICP
[Open3D DEBUG] ICP Iteration #0: Fitness 0.0420, RMSE 0.0623
[Open3D DEBUG] Residual : 2.78e-03 (# of elements : 214327)
[Open3D DEBUG] ICP Iteration #1: Fitness 0.0458, RMSE 0.0548
[Open3D DEBUG] Residual : 1.84e-03 (# of elements : 233554)
[Open3D DEBUG] ICP Iteration #2: Fitness 0.0507, RMSE 0.0596
[Open3D DEBUG] Residual : 2.24e-03 (# of elements : 258707)
[Open3D DEBUG] ICP Iteration #3: Fitness 0.0734, RMSE 0.0900
[Open3D DEBUG] Residual : 5.52e-03 (# of elements : 374359)
[Open3D DEBUG] ICP Iteration #4: Fitness 0.7106, RMSE 0.0983
[Open3D DEBUG] Residual : 8.88e-03 (# of elements : 3626693)
[Open3D DEBUG] ICP Iteration #5: Fitness 0.7165, RMSE 0.0279
[Open3D DEBUG] Residual : 1.02e-04 (# of elements : 3656878)
[Open3D DEBUG] ICP Iteration #6: Fitness 0.7181, RMSE 0.0276
[Open3D DEBUG] Residual : 9.12e-05 (# of elements : 3664930)
[Open3D DEBUG] ICP Iteration #7: Fitness 0.7193, RMSE 0.0274

## Pose graph

In [10]:
print("Full registration ...")
max_correspondence_distance_coarse = 0.2#voxel_size * 15
max_correspondence_distance_fine1  = 0.01#voxel_size * 1.5
max_correspondence_distance_fine2  = 0.005#voxel_size * 1.5
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    pose_graph = full_registration_with_pairs(
        pcds_down, pair_info,
        max_correspondence_distance_coarse,
        max_correspondence_distance_fine1,
        max_correspondence_distance_fine2
    )

Full registration ...
----SOURCE, TARGET = 1, 2
Coarse alignment
Apply point-to-plane ICP
[Open3D DEBUG] ICP Iteration #0: Fitness 0.0976, RMSE 0.0941
[Open3D DEBUG] Residual : 5.03e-03 (# of elements : 253991)
[Open3D DEBUG] ICP Iteration #1: Fitness 0.9536, RMSE 0.1347
[Open3D DEBUG] Residual : 1.75e-02 (# of elements : 2482385)
[Open3D DEBUG] ICP Iteration #2: Fitness 0.9612, RMSE 0.0240
[Open3D DEBUG] Residual : 1.51e-04 (# of elements : 2502241)
[Open3D DEBUG] ICP Iteration #3: Fitness 0.9615, RMSE 0.0240
[Open3D DEBUG] Residual : 1.48e-04 (# of elements : 2502995)
[Open3D DEBUG] ICP Iteration #4: Fitness 0.9655, RMSE 0.0267
[Open3D DEBUG] Residual : 2.77e-04 (# of elements : 2513538)
[Open3D DEBUG] ICP Iteration #5: Fitness 0.9690, RMSE 0.0242
[Open3D DEBUG] Residual : 1.69e-04 (# of elements : 2522576)
[Open3D DEBUG] ICP Iteration #6: Fitness 0.9679, RMSE 0.0222
[Open3D DEBUG] Residual : 7.56e-05 (# of elements : 2519778)
[Open3D DEBUG] ICP Iteration #7: Fitness 0.9675, RMSE 0.0

Open3D uses the function `global_optimization` to perform pose graph optimization. Two types of optimization methods can be chosen: `GlobalOptimizationGaussNewton` or `GlobalOptimizationLevenbergMarquardt`. The latter is recommended since it has better convergence property. Class `GlobalOptimizationConvergenceCriteria` can be used to set the maximum number of iterations and various optimization parameters.

Class `GlobalOptimizationOption` defines a couple of options. `max_correspondence_distance` decides the correspondence threshold. `edge_prune_threshold` is a threshold for pruning outlier edges. `reference_node` is the node id that is considered to be the global space.

In [12]:
# max_correspondence_distance_fine = 0.005
print("Optimizing PoseGraph ...")
option = o3d.pipelines.registration.GlobalOptimizationOption(
    max_correspondence_distance=max_correspondence_distance_fine2,
    edge_prune_threshold=0.25,
    reference_node=0)
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    o3d.pipelines.registration.global_optimization(
        pose_graph,
        o3d.pipelines.registration.GlobalOptimizationLevenbergMarquardt(),
        o3d.pipelines.registration.GlobalOptimizationConvergenceCriteria(),
        option)

Optimizing PoseGraph ...
[Open3D DEBUG] Validating PoseGraph - finished.
[Open3D DEBUG] [GlobalOptimizationLM] Optimizing PoseGraph having 16 nodes and 15 edges.
[Open3D DEBUG] Line process weight : 24.566295
[Open3D DEBUG] [Initial     ] residual : 4.407961e+06, lambda : 6.978360e+03
[Open3D DEBUG] [Iteration 00] residual : 7.201922e+03, valid edges : 0, time : 0.000 sec.
[Open3D DEBUG] [Iteration 01] residual : 5.498873e+02, valid edges : 0, time : 0.000 sec.
[Open3D DEBUG] [Iteration 02] residual : 1.724519e+01, valid edges : 0, time : 0.000 sec.
[Open3D DEBUG] [Iteration 03] residual : 1.196239e-01, valid edges : 0, time : 0.000 sec.
[Open3D DEBUG] [Iteration 04] residual : 1.182515e-04, valid edges : 0, time : 0.000 sec.
[Open3D DEBUG] [Iteration 05] residual : 1.463680e-08, valid edges : 0, time : 0.000 sec.
[Open3D DEBUG] Current_residual < 1.000000e-06
[Open3D DEBUG] [GlobalOptimizationLM] total time : 0.001 sec.
[Open3D DEBUG] [GlobalOptimizationLM] Optimizing PoseGraph having

The global optimization performs twice on the pose graph. The first pass optimizes poses for the original pose graph taking all edges into account and does its best to distinguish false alignments among uncertain edges. These false alignments have small line process weights, and they are pruned after the first pass. The second pass runs without them and produces a tight global alignment. In this example, all the edges are considered as true alignments, hence the second pass terminates immediately.

## Visualize optimization
The transformed point clouds are listed and visualized using `draw_geometries`.

In [13]:
# print("Transform points and display")
# for point_id in range(len(pcds_down)):
#     pcds_down[point_id].transform(pose_graph.nodes[point_id].pose)
# o3d.visualization.draw_geometries(pcds_down,
#                                   zoom=0.3412,
#                                   front=[0.4257, -0.2125, -0.8795],
#                                   lookat=[2.6172, 2.0475, 1.532],
#                                   up=[-0.0694, -0.9768, 0.2024])

## Make a combined point cloud
`PointCloud` has a convenience operator `+` that can merge two point clouds into a single one. In the code below, the points are uniformly resampled using `voxel_down_sample` after merging. This is recommended post-processing after merging point clouds since it can relieve duplicated or over-densified points.

In [51]:
# inliner_pcd_combined_down = combine_point_cloud(deepcopy(pcds_down), pose_graph)
inliner_pcd_combined_down = combine_point_cloud(deepcopy(pcd_pairs[1:]), pose_graph)
# write_array2txt(
#     os.path.join(data_dir, f'inliers_combined_with_ids_down.txt'),
#     inliner_pcd_combined_down)


In [52]:
# write a slice
l = 3.0
thickness = 0.01
ind = ((l <= inliner_pcd_combined_down[:,2]) & (inliner_pcd_combined_down[:,2] <= l + thickness))
write_array2txt(
    os.path.join(data_dir, f'slice_3_down_optimize_.txt'),
    inliner_pcd_combined_down[ind,:]
)

In [17]:
inliner_pcd_combined = combine_point_cloud(pcds_full, pose_graph)
# pcds = load_point_clouds(path_,voxel_size)
# pcd_combined = o3d.geometry.PointCloud()
# pcd_full = np.empty(shape=[0,4])
# for point_id in range(len(pcds)):
#     pcds[point_id].transform(pose_graph.nodes[point_id].pose)
#     A = np.asarray(pcds[point_id].points)
#     B = np.c_[ A, np.ones( A.shape[0] )*point_id ]
#     pcd_full = np.r_[pcd_full,B]

In [18]:
write_array2txt(
    os.path.join(data_dir, f'inliers_combined_with_ids.txt'),
    inliner_pcd_combined)


In [19]:
o3d.io.write_point_cloud(
    os.path.join(data_dir, f'all_down.pcd'),
    pcd, print_progress=True
)

NameError: name 'pcd' is not defined

In [ ]:
inliner_pcd_combined.shape

(642487958, 4)

In [ ]:
# write a slice
l = 3.0
thickness = 0.01
ind = ((l <= inliner_pcd_combined[:,2]) & (inliner_pcd_combined[:,2] <= l + thickness))
write_array2txt(
    os.path.join(data_dir, f'slice_inliers_combined_with_ids.txt'),
    inliner_pcd_combined[ind,:]
)

NameError: name 'inliner_pcd_combined' is not defined

In [63]:
ind = inliner_pcd_combined[:,-1]==5
inliner_pcd_combined[ind,2].min()

-0.002986272441479922

In [56]:
inliner_pcd_combined[ind,-1].max()

7.0

In [32]:
write_array2txt(
    os.path.join(data_dir, f'tank_{tank_name}_without004_no_ids.txt'),
    inliner_pcd_combined[:,:3])

In [17]:
p8 = pcds_down[8]
p9 = pcds_down[9]

In [21]:
p7 = pcds_down[7]

In [23]:
min(p9.compute_point_cloud_distance(p7))

0.0005837807807712218

In [18]:
dist89 = p8.compute_point_cloud_distance(p9)

In [20]:
min(dist89)

0.5343259772835293

## Handle outlier cylinders

In [34]:
cyl_combined = Cylinder(
    metadata = {
        'ptc_filepath': os.path.join(data_dir, f'tank_{tank_name}_without004_no_ids.txt'), #'inliers_combined.txt'),
        'tank_id': 1,
        'station': Station(name=f'tank_{tank_name}_without004_no_ids')
    }
)
out_cyls.insert(0,cyl_combined)

In [41]:
# read points
pcds_down, pcds_full = load_point_clouds(out_cyls)
# pose graph
print("Full registration ...")
max_correspondence_distance_coarse = 1#voxel_size * 15
max_correspondence_distance_fine = 0.01#voxel_size * 1.5
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    pose_graph = full_registration(pcds_down,
                                   max_correspondence_distance_coarse,
                                   max_correspondence_distance_fine)
# optimize
print("Optimizing PoseGraph ...")
option = o3d.pipelines.registration.GlobalOptimizationOption(
    max_correspondence_distance=max_correspondence_distance_fine,
    edge_prune_threshold=0.25,
    reference_node=0)
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    o3d.pipelines.registration.global_optimization(
        pose_graph,
        o3d.pipelines.registration.GlobalOptimizationLevenbergMarquardt(),
        o3d.pipelines.registration.GlobalOptimizationConvergenceCriteria(),
        option)
# make combined
pose_graph.nodes.pop(0)
outlier_pcd_combined = combine_point_cloud(pcds_full[1:], pose_graph)
# inlier + outlier
pcd_combined = np.r_[inliner_pcd_combined, outlier_pcd_combined]
# pcd_combined = combine_point_cloud(pcds_full, pose_graph)
# write
write_array2txt(
    data_dir + '\complete_combined_with_ids.txt',
    pcd_combined)

D:\REGISTRATION DATA\External\JB-595 117ft EXT - FARO\tank_1\tank_1_without004_no_ids.txt
D:\REGISTRATION DATA\External\JB-595 117ft EXT - FARO\tank_1\004.txt
Full registration ...
----SOURCE 0----
===== source_id=0, target_id=1 =====
Apply point-to-plane ICP
Coarse alignment
[Open3D DEBUG] ICP Iteration #0: Fitness 0.3063, RMSE 0.2806
[Open3D DEBUG] Residual : 3.11e-02 (# of elements : 1849886)
[Open3D DEBUG] ICP Iteration #1: Fitness 0.3055, RMSE 0.2698
[Open3D DEBUG] Residual : 2.67e-02 (# of elements : 1845158)
[Open3D DEBUG] ICP Iteration #2: Fitness 0.3056, RMSE 0.2698
[Open3D DEBUG] Residual : 2.75e-02 (# of elements : 1845523)
[Open3D DEBUG] ICP Iteration #3: Fitness 0.3056, RMSE 0.2703
[Open3D DEBUG] Residual : 2.80e-02 (# of elements : 1845495)
[Open3D DEBUG] ICP Iteration #4: Fitness 0.3055, RMSE 0.2703
[Open3D DEBUG] Residual : 2.81e-02 (# of elements : 1845070)
[Open3D DEBUG] ICP Iteration #5: Fitness 0.3056, RMSE 0.2699
[Open3D DEBUG] Residual : 2.84e-02 (# of elements : 